# ML-03 â€” Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SammyFarrelZebua/flyrank-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **CTR / Engagement Opportunity Scoring**. We frame this as a **Ranking / scoring** task (which can be solved by training a regression model to predict expected CTR or engagement rate, and then scoring pages by their performance gap relative to the expected rate).

- **Why this task type?**
  The business objective is to help Content Editors and SEO Specialists prioritize which pages to review and refresh first (i.e., "which ones first?"). A binary classification (decline vs. no decline) is too coarse, and a plain regression of CTR/engagement alone doesn't account for traffic volume or position.
  By predicting the *expected* CTR/engagement based on page features and ranking pages by the *Click Opportunity Score* (defined as `(Expected_CTR - Actual_CTR) * Impressions`), we provide a sorted queue that prioritizes pages where updates will yield the highest traffic recovery or increase.

- **The One-Paragraph Frame:**
  For **Content Editors and SEO Specialists**, deciding **which underperforming pages to prioritize for content audit and refresh first**, we will build a **ranked opportunity queue (scoring model)** from **Google Search Console and Google Analytics 4 trailing 90-day search/user behavior metrics**, predicting/scoring **expected CTR (and the resulting click opportunity score gap)** measured by **Mean Absolute Error (MAE) and Precision@50**. A wrong call costs **wasted editor hours reviewing pages that do not need changes or are already optimized**. A plain rule isn't enough because **CTR is non-linearly dependent on the average search position, and user engagement depends on complex interactions of word count, content age, type, and intent**. We will claim only **observed, directional, and decision-support** results.


In [1]:
import os
import pandas as pd
import numpy as np

# Load starter dataset
data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Filter for pages with reliable search data (impressions >= 100)
# Note: This also filters out all 1,205 rows with avg_position == 0 (no position data)
df_clean = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# Compute expected CTR baseline by position tier
# Note: In w01, the raw mean for top_3 was 1.48% because it was inflated by pages with very low impressions 
# (e.g. 1 impression and 1 click, yielding 100% CTR). Filtering for impressions >= 100 removes these noisy 
# outliers, resulting in a clean top_3 mean CTR of 0.334%.
tier_ctr = df_clean.groupby("position_tier")["ctr"].mean().to_dict()
df_clean["expected_ctr"] = df_clean["position_tier"].map(tier_ctr)

# Calculate click opportunity score (estimated clicks lost)
df_clean["ctr_gap"] = df_clean["expected_ctr"] - df_clean["ctr"]
df_clean["opportunity_score"] = df_clean["ctr_gap"] * df_clean["impressions_90d"] / 100.0

# Show the top 5 highest opportunity pages in the ranked queue
print("Top 5 Highest Opportunity Pages (Ranked Queue Baseline):")
print(df_clean[["content_id", "position_tier", "impressions_90d", "ctr", "expected_ctr", "opportunity_score"]].sort_values(by="opportunity_score", ascending=False).head(5))


Top 5 Highest Opportunity Pages (Ranked Queue Baseline):
                 content_id position_tier  impressions_90d   ctr  \
6653   content_5fe46e04994d        page_1           517715  0.14   
26844  content_8c19996aa890         top_3           509252  0.15   
3394   content_36ff89c8214e        page_1           295097  0.05   
7678   content_8451fc6f034d         top_3           272144  0.03   
7445   content_c8e9d6ab9013        page_1           208678  0.00   

       expected_ctr  opportunity_score  
6653       0.354760        1111.842887  
26844      0.334128         937.673382  
3394       0.354760         899.336564  
7678       0.334128         827.664961  
7445       0.354760         740.305328  


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target we predict is the **expected CTR** (specifically `ctr` from Google Search Console, or `engagement_rate` from Google Analytics 4).

- **Where does the label come from?**
  It is a purely **observed outcome** measured over a trailing 90-day window, not a defined heuristic or rule. It represents actual user behavior (clicks relative to impressions or engaged sessions relative to sessions).
- **Target vs. Proxy:**
  Our final ranking score (Click Opportunity Score) is a derived proxy metric (`Expected_CTR - Actual_CTR`), but the underlying model trains on the *observed CTR* to learn the baseline relationship between page properties (position, search volume, word count, age, etc.) and CTR.
  We explicitly avoid using `trend_direction` or `trend_pct` as features, as they would create target leakage.


In [2]:
# Show distribution of the target variable (CTR) for clean data
print("Observed CTR distribution (percent):")
print(df_clean["ctr"].describe())

# Check that the target is non-trivial and has variance
print(f"\nTarget variance: {df_clean['ctr'].var():.4f}")


Observed CTR distribution (percent):
count    22006.000000
mean         0.257281
std          0.402118
min          0.000000
25%          0.000000
50%          0.140000
75%          0.340000
max         11.760000
Name: ctr, dtype: float64

Target variance: 0.1617


## 3. Success metric

*One metric you can defend. What number means 'good'?*

We define two levels of success metrics:

1. **Model Metric (Regression)**:
   **Mean Absolute Error (MAE)** of the predicted CTR. Our baseline model (predicting the mean CTR of the position tier) achieves an MAE of **0.2257%**. A successful ML model must achieve an MAE of **< 0.20%** on the client-holdout split.

2. **Business Metric (Ranking)**:
   **Precision@50** of the opportunity queue. We define a page as "truly underperforming" if its actual CTR is in the bottom 25% of pages within its `position_tier`.
   - A random selection of pages with impressions >= 100 yields a **Precision@50 of 28.0%**.
   - Our baseline opportunity scoring yields a **Precision@50 of 44.0%**.
   - A successful ML-based opportunity ranking must achieve a **Precision@50 of > 60%** (a 2.1x lift over random and a 36% improvement over the simple baseline).


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Define ground truth underperformance (bottom 25% CTR in tier)
tier_25th = df_clean.groupby("position_tier")["ctr"].quantile(0.25).to_dict()
df_clean["ctr_25th"] = df_clean["position_tier"].map(tier_25th)
df_clean["is_underperforming"] = (df_clean["ctr"] <= df_clean["ctr_25th"]).astype(int)

# Calculate baseline metrics
mae = mean_absolute_error(df_clean["ctr"], df_clean["expected_ctr"])
rmse = np.sqrt(mean_squared_error(df_clean["ctr"], df_clean["expected_ctr"]))

# Precision@50 for baseline opportunity score
p_50_opp = df_clean.sort_values(by="opportunity_score", ascending=False).head(50)["is_underperforming"].mean() * 100

# Precision@50 for random
p_50_rand = df_clean.sample(frac=1, random_state=42).head(50)["is_underperforming"].mean() * 100

print(f"Baseline MAE (predicting tier mean): {mae:.4f}%")
print(f"Baseline RMSE (predicting tier mean): {rmse:.4f}%")
print(f"Baseline Opportunity Queue Precision@50: {p_50_opp:.1f}%")
print(f"Random Selection Precision@50: {p_50_rand:.1f}%")


Baseline MAE (predicting tier mean): 0.2257%
Baseline RMSE (predicting tier mean): 0.3906%
Baseline Opportunity Queue Precision@50: 44.0%
Random Selection Precision@50: 28.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content item (page) for a given client over the trailing 90 days**.

- **Grain:**
  In the starter dataset, each row corresponds to one unique `content_id` (a page) belonging to one `client_id` (a site).
- **Features and Target:**
  The features consist of keyword metadata (search volume, competition), content properties (word count, content age, freshness), and traffic context (impressions, avg_position).
  The target is the observed `ctr`.


In [4]:
# Show the unit of analysis as a slice of the dataframe (features + target)
cols_to_show = [
    "content_id", "client_id", "content_type", "word_count", 
    "content_age_days", "avg_position", "impressions_90d", "ctr"
]
print("Unit of analysis (Sample rows from starter dataset):")
print(df_clean[cols_to_show].head(5))
print(f"\nDataFrame shape for clean analysis unit: {df_clean[cols_to_show].shape}")


Unit of analysis (Sample rows from starter dataset):
             content_id          client_id     content_type  word_count  \
0  content_304f48230142  client_f369cb89fc  keyword article      3221.0   
1  content_a1fb4e703a9e  client_4e07408562  keyword article      2481.0   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article      3515.0   
3  content_331d6c4de07b  client_19581e27de  keyword article         NaN   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article      2803.0   

   content_age_days  avg_position  impressions_90d   ctr  
0               187          10.6             3803  0.76  
1               445          20.3            15320  0.05  
2               141          36.5            12581  0.09  
3               463           6.2            11751  0.49  
4               263          44.0            19140  0.13  

DataFrame shape for clean analysis unit: (22006, 8)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule (e.g., "review all pages with CTR < 1.0%") fails for two reasons:

1. **Non-linear Relationship with Position**:
   CTR is highly dependent on search ranking (`avg_position`). A page at position 1.5 with 1.0% CTR is underperforming significantly (mean top_3 CTR is 1.48%), whereas a page at position 15 with 0.5% CTR is performing exceptionally well (mean deep CTR is 0.15%). A flat threshold would flag the wrong pages.
2. **Multi-variable Interactions**:
   CTR is also influenced by other factors such as `content_type`, `search_volume`, `word_count`, and `content_age_days`. These variables interact in complex, non-linear ways that cannot be mapped with simple nested if-statements. An ML model can capture these multi-dimensional patterns to predict a more accurate, personalized expected CTR for each page.


In [ ]:
# Group by average position (rounded to integer) and show average CTR
df_clean["rounded_position"] = df_clean["avg_position"].round().astype(int)
pos_ctr = df_clean.groupby("rounded_position")["ctr"].agg(["mean", "count"])
print("Average CTR by rounded position (showing first 10 ranks):")
print(pos_ctr.head(10))

# This non-linear decay shows why a fixed rule is inadequate


Average CTR by rounded position (showing first 10 ranks):
                      mean  count
rounded_position                 
0                 0.087500      4
1                 0.122391     46
2                 0.342680    291
3                 0.420536    466
4                 0.443583   1030
5                 0.430782   1253
6                 0.356057   1689
7                 0.329558   1359
8                 0.292688   1529
9                 0.309185    957


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
